## Import libraries

In [1]:
import numpy as np
from pathlib import Path
import tifffile
from careamics.model_io import load_pretrained
from careamics.utils.metrics import scale_invariant_psnr
from careamics import CAREamist
from careamics.config import create_care_configuration

/opt/.pixi/envs/napari-careamics-default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the data

In [2]:
data_split = "U2OS_splits_n100_seed42"
prepared_dir = Path(f"../data_careamics/{data_split}")

# train_path = prepared_dir / "train" / "low"
# train_target_path = prepared_dir / "train" / "high"
# val_path = prepared_dir / "validation" / "low"
# val_target_path = prepared_dir / "validation" / "high"

test_path = prepared_dir / "test" / "low"
test_target_path = prepared_dir / "test" / "high"

In [3]:
# Show two images
# test_GT_path = root_path / "test" / "GT"
test_GT_files = list(test_target_path.rglob("*.tif"))
test_GT_files.sort()

# test_low_path = root_path / "test" / "low"
test_low_files = list(test_path.rglob("*.tif"))
test_low_files.sort()

test_GT = [tifffile.imread(f) for f in test_GT_files]

## Load the model

In [4]:
model_path = "../scripts/checkpoints/last-v9.ckpt"

In [5]:
careamist = CAREamist(source=model_path)

No working directory provided. Using current working directory: /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/CAREamics/scripts.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [6]:
print(careamist.cfg)

{'algorithm_config': {'algorithm': 'care',
                      'loss': 'mae',
                      'lr_scheduler': {'name': 'ReduceLROnPlateau',
                                       'parameters': {}},
                      'model': {'architecture': 'UNet',
                                'conv_dims': 2,
                                'depth': 2,
                                'final_activation': 'None',
                                'in_channels': 1,
                                'independent_channels': True,
                                'n2v2': False,
                                'num_channels_init': 32,
                                'num_classes': 1,
                                'use_batch_norm': True},
                      'optimizer': {'name': 'Adam', 'parameters': {}}},
 'data_config': {'axes': 'YX',
                 'batch_size': 512,
                 'data_type': 'tiff',
                 'image_means': ['1.0162066e+01'],
                 'image_stds': ['1.

## Predict with CAREamics

Prediction is done with a pretrained `CAREamist` stored in your project, and we can use the test set.

In [7]:
prediction = careamist.predict(source=test_path)

You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-871230e5-4f84-5f7b-a288-f2c24ae0169b]
SLURM auto-requeueing enabled. Setting signal handlers.
/opt/.pixi/envs/napari-careamics-default/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: |                       | 10/? [00:02<00:00,  3.93it/s]


## Compute metrics


<font size = 4>This section will display SSIM maps and RSE maps as well as calculating total SSIM, NRMSE and PSNR metrics for all the images provided in the "Source_QC_folder" and "Target_QC_folder" !

<font size = 4>**1. The SSIM (structural similarity) map**

<font size = 4>The SSIM metric is used to evaluate whether two images contain the same structures. It is a normalized metric and an SSIM of 1 indicates a perfect similarity between two images. Therefore for SSIM, the closer to 1, the better. The SSIM maps are constructed by calculating the SSIM metric in each pixel by considering the surrounding structural similarity in the neighbourhood of that pixel (currently defined as window of 11 pixels and with Gaussian weighting of 1.5 pixel standard deviation, see our Wiki for more info).

<font size=4>**mSSIM** is the SSIM value calculated across the entire window of both images.

<font size=4>**The output below shows the SSIM maps with the mSSIM**

<font size = 4>**2. The RSE (Root Squared Error) map**

<font size = 4>This is a display of the root of the squared difference between the normalized predicted and target or the source and the target. In this case, a smaller RSE is better. A perfect agreement between target and prediction will lead to an RSE map showing zeros everywhere (dark).


<font size =4>**NRMSE (normalised root mean squared error)** gives the average difference between all pixels in the images compared to each other. Good agreement yields low NRMSE scores.

<font size = 4>**PSNR (Peak signal-to-noise ratio)** is a metric that gives the difference between the ground truth and prediction (or source input) in decibels, using the peak pixel values of the prediction and the MSE between the images. The higher the score the better the agreement.

<font size=4>**The output below shows the RSE maps with the NRMSE and PSNR values.**





### PSNR

In [8]:
psnrs = np.zeros((len(prediction), 1))

for i, (pred, gt) in enumerate(zip(prediction, test_GT)):
    psnrs[i] = scale_invariant_psnr(gt, pred.squeeze())

print(f"PSNR: {psnrs.mean():.2f} +/- {psnrs.std():.2f}")

PSNR: 29.83 +/- 3.27
